# Etapa 4 — Avaliação e Golden Set

Duas entregas do enunciado: (1) métricas de avaliação do modelo, além da
`conversion_rate` já vista na Etapa 3; (2) Golden Set de 5 clientes reais,
mostrando a recomendação da política treinada e se ela faz sentido.

## Passo 1 — Regret médio por rodada

Regret = `taxa_oráculo - taxa_política`. Quantifica quão longe cada
política fica do braço ótimo (`cellular`, 14,74% — achado da Etapa 3).
Complementa a `conversion_rate` já calculada.

In [1]:
from pathlib import Path

from datathon_mlet.data_prep import load_clean_dataset, prepare_features
from datathon_mlet.policies import FixedPolicy, ThompsonSamplingPolicy
from datathon_mlet.replay import run_replay

CLEAN_DATASET_PATH = Path("../data/processed/bank_marketing_clean.parquet")

df_clean = load_clean_dataset(CLEAN_DATASET_PATH)
prepared = prepare_features(df_clean)

ORACLE_CONVERSION_RATE = 0.14737607099143207  # cellular, achado da Etapa 3

baseline_policy = FixedPolicy(arm="telephone")
baseline_result = run_replay(prepared.action, prepared.reward, baseline_policy, seed=0)

ts_policy = ThompsonSamplingPolicy(arms=["cellular", "telephone"])
ts_result = run_replay(prepared.action, prepared.reward, ts_policy, seed=0)

baseline_regret = ORACLE_CONVERSION_RATE - baseline_result.conversion_rate
ts_regret = ORACLE_CONVERSION_RATE - ts_result.conversion_rate

baseline_regret, ts_regret

(0.09506285642083914, 0.00034323058428784026)

## Passo 2 — Golden Set (5 clientes reais)

5 linhas reais do dataset, `poutcome` variado (`success`/`failure`/`nonexistent`), recomendação de `ts_policy.recommend()` (já treinada pelo `run_replay` do passo 1) comparada com o histórico real.

**Nota honesta:** como o bandit é não-contextual (decisão da Etapa 3), a recomendação é a mesma pra todos os 5 — não personaliza por perfil de cliente, só aprendeu qual braço é melhor em agregado.

In [ ]:
golden_set_indices = (
    df_clean[df_clean["poutcome"] == "success"].index[:2].tolist()
    + df_clean[df_clean["poutcome"] == "failure"].index[:2].tolist()
    + df_clean[df_clean["poutcome"] == "nonexistent"].index[:1].tolist()
)

golden_set = df_clean.loc[
    golden_set_indices,
    ["age", "job", "marital", "education", "poutcome", "contact", "y"],
].copy()
golden_set["recomendacao_politica"] = ts_policy.recommend()
golden_set

## Conclusão

**Métricas de avaliação (Etapa 3 + regret desta etapa):**

| Política | Conversão | Regret vs oráculo (`cellular`, 14,74%) |
|---|---|---|
| Baseline (`telephone` fixo) | 5,23% | 9,51 p.p. |
| Thompson Sampling | 14,70% | 0,03 p.p. |

Thompson Sampling fica a 0,03 ponto percentual do braço ótimo em retrospecto — praticamente ótimo. O baseline ingênuo fica quase 10 pontos atrás.

**Golden Set (5 clientes reais, `poutcome` variado):** a política recomenda `cellular` para os 5 — esperado, já que o bandit é não-contextual (decisão da Etapa 3) e aprendeu apenas a taxa agregada por canal, não personaliza por cliente. Cada recomendação individual faz sentido no nível agregado (é o canal historicamente melhor), mas não captura variação de caso a caso: o cliente `idx 24019` teve campanha anterior malsucedida (`poutcome=failure`) e foi contatado por `telephone`, ainda assim converteu (`y=yes`) — ruído individual que uma política contextual (extensão futura, contexto já preparado na Etapa 2) poderia capturar melhor.

**Limitações (adicionais às já documentadas na Etapa 3):** o Golden Set não testa generalização por cliente — testa se a política aprendeu corretamente o agregado. Uma avaliação de personalização exigiria a formulação contextual descartada por simplicidade na Etapa 3.